# Day 2 — HOL 3: Code Versioning — Databricks Repos + GitHub

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 3 — Code Versioning Concepts (Git basics, dev/test/prod branching) |
| **Duration** | 2 hours |
| **Output** | A GitHub repo connected to Databricks Repos, with a `dev` branch and one real commit pushed |

### Learning Objectives
- Create a GitHub repository and a Personal Access Token
- Connect that repository to Databricks Repos
- Create a `dev` branch and confirm you're working on it
- Build a shared config notebook using the Unity Catalog External Location pattern from HOL 1 (no storage key)
- Commit and push a real change from Databricks to GitHub

---
**Instructions:** Phases A–C are GitHub/Databricks UI steps — read carefully, screenshot as you go. Phase D has real notebook cells to run. Phase E is the commit/push workflow you'll repeat every day for the rest of the bootcamp.

---
## Phase A — Create Your GitHub Repository

### A1 — Create the repo

```
1. Go to github.com and log in (create a free account if needed)
2. Click + New repository
3. Name it: globalmart-lakehouse
4. Set it to Private
5. Check "Add a README file"
6. Click Create repository
```

### A2 — Generate a Personal Access Token

```
1. github.com → Settings → Developer Settings → Personal Access Tokens → Tokens (classic)
2. Click Generate new token (classic)
3. Name it, set expiry to 90 days, check the "repo" scope
4. Click Generate token → COPY the token immediately (you cannot see it again)
```

> **Common mistake:** closing the token page before copying it. If that happens, just generate a new one — there's no way to recover the old value.

---
## Phase B — Connect GitHub to Databricks

### B1 — Link your GitHub account

```
1. In Databricks, click your username (top right) → Settings
2. Go to Linked Accounts → Git Integration
3. Select GitHub as the provider
4. Paste your Personal Access Token from A2
5. Enter your GitHub username
6. Click Save
```

### B2 — Clone the repo into Databricks

```
1. In the left sidebar, click Repos
2. Click Add Repo
3. Paste your GitHub repo URL: https://github.com/YOUR_USERNAME/globalmart-lakehouse
4. Click Create Repo
```

**Verify:** your repo appears in Databricks under Repos, with a README.md file inside it.

---
## Phase C — Create Your `dev` Branch

**Goal:** never write code directly on `main` — same discipline as the dev → test → prod pattern from ILT 3.

```
1. Open your repo in Databricks Repos
2. Click the Git button (branch icon, top of the notebook/repo view)
3. Click Create Branch → name it: dev
4. Confirm the branch selector now shows "dev"
```

**Verify:** the branch indicator in the Databricks Repos UI reads `dev`, not `main`.

In [ ]:
# ─── C1: Verify you are working inside a Git repo, on the dev branch ─────────
# If the notebook path contains '/Repos/' it is Git-tracked.
# If it contains '/Users/' it is NOT in a repo — go back to Phase B.

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"Current notebook path: {notebook_path}")

if "/Repos/" in notebook_path:
    print("This notebook IS inside a Databricks Repo — Git tracking is active.")
else:
    print("This notebook is NOT inside a Repo. Move it into your Repos folder (Phase B) before continuing.")

---
## Phase D — Build the Shared Config Notebook

**Goal:** one `config/setup` notebook that every future notebook in this repo calls with `%run`, instead of repeating ADLS connection code everywhere — and it uses the **External Location** pattern from HOL 1, not a hardcoded key.

### D1 — Create the folder and notebook

```
1. Inside your repo (on the dev branch), create a folder called config
2. Inside config/, create a new notebook called setup
```

Copy the cell below into that `config/setup` notebook, filling in your own storage account name from Day 1.

In [ ]:
# ============================================================
# config/setup — shared connection config for the GlobalMart repo
#
# Uses the Unity Catalog External Location you created in HOL 1 —
# no storage key anywhere in this notebook or in git history.
# Every other notebook in this repo loads these variables with:
#     %run ../config/setup
# ============================================================

STORAGE_ACCOUNT_NAME = "YOUR_STORAGE_ACCOUNT_NAME"   # ← your storage account from Day 1
CONTAINER_NAME        = "YOUR_CONTAINER_NAME"        # ← your container from Day 1

# No spark.conf.set() needed — your External Location from HOL 1 already
# covers this path; Unity Catalog handles authentication for us.
BASE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"
RAW_PATH  = f"{BASE_PATH}/raw-data"   # source CSV/JSON files land here, one subfolder per entity
                                       # (customers/, addresses/, payments/, payment_methods/, products/, returns/)

# Bronze/Silver/Gold are NOT ADLS folders — they're schemas inside your Unity Catalog
# catalog, referenced by name only (no path at all): spark.table(f"{CATALOG}.bronze.customers")
CATALOG = "YOUR_CATALOG"   # ← your catalog from Day 1

print("GlobalMart config loaded successfully.")
print(f"  Raw data path  : {RAW_PATH}")
print(f"  Catalog        : {CATALOG}")

In [ ]:
# ─── D3: Confirm %run picks up the shared config from another notebook ──────
# Run this from a DIFFERENT notebook in the same repo (not config/setup itself)
# to prove the pattern works. Adjust the relative path if your notebook lives
# somewhere other than directly beside the config/ folder.

# %run ../config/setup

# After the line above runs, RAW_PATH / CATALOG are both available in this
# notebook without redefining them.
print("Uncomment the %run line above once config/setup.ipynb exists in your repo.")

---
## Phase E — Commit and Push

**Goal:** get `config/setup` from your local Databricks Repo into GitHub, on the `dev` branch. This is the exact workflow you'll repeat every day for the rest of the bootcamp.

```
Step 1: In Databricks Repos, click the Git button on your repo
Step 2: You'll see config/setup.ipynb listed as a changed file
Step 3: Stage the file
Step 4: Write a commit message:
        "Add shared ADLS config notebook using External Location auth"
Step 5: Click Commit
Step 6: Click Push
Step 7: Go to github.com → your repo → switch to the dev branch
Step 8: Confirm config/setup.ipynb appears there
```

### Good vs Bad Commit Messages (recap from ILT 3)

| Bad | Good |
|---|---|
| `update` | `Add null check for customer email in silver layer` |
| `fix` | `Fix date format in orders table from MM/DD to YYYY-MM-DD` |
| `changes` | `Add shared ADLS config notebook using External Location auth` |

**Rule:** your commit message should tell a teammate exactly what changed and why, without them having to open the diff.

---
## Submission Checklist

> ⚠️ Replace any real token/secret values with placeholders before uploading this notebook. Personal Access Tokens should never appear in a notebook cell — they're entered once in the Databricks UI (Phase B1), never pasted into code.

```
Submission Checklist
────────────────────────────────────────────────────────────────
✅ GitHub repo created: globalmart-lakehouse (Private)
✅ Personal Access Token generated and saved in Databricks Git Integration
✅ Repo cloned into Databricks under Repos
✅ dev branch created and confirmed active
✅ config/setup notebook created with your own STORAGE_ACCOUNT_NAME
✅ config/setup uses External Location auth — no storage key in the notebook
✅ Committed config/setup with a clear, descriptive message
✅ Pushed to GitHub — confirmed the file appears under the dev branch
✅ Screenshot: Databricks Repos view showing the dev branch active
✅ Screenshot: GitHub repo showing config/setup.ipynb under dev
────────────────────────────────────────────────────────────────
```

---

## What Comes Next

| Session | Topic |
|---------|-------|
| **Day 3** | Autoloader + REST API + GraphDB ingestion patterns |

From tomorrow, every notebook you write gets committed to this repo — by the end of the bootcamp your GitHub will show a complete, professional data engineering project.